# Epic 7 -- Spectral Reflectance Model

This notebook explores the material reflectance model used in Chromasense
multi-spectral imaging.  Each semiconductor packaging material has a
characteristic spectral fingerprint that allows it to be identified
across the Chromasense wavelength bands.

## Topics covered

1. Default materials and their spectral signatures
2. Reflectance interpolation between calibrated wavelengths
3. Comparing material spectra visually
4. Custom spectral configurations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from udm_epic7.spectral.wavelength_model import (
    default_spectral_config,
    material_reflectance,
    SpectralConfig,
)

cfg = default_spectral_config()
print(f"Wavelengths: {cfg.wavelengths}")
print(f"Materials:   {cfg.material_names}")

---
## 1. Material Spectral Signatures

Let's plot the reflectance curve for each default material across a dense
wavelength range.  The interpolation function fills in values between the
four calibrated Chromasense wavelengths.

In [ ]:
wl_dense = np.linspace(430, 870, 200)

fig, ax = plt.subplots(figsize=(10, 6))
for mat in cfg.material_names:
    refs = [material_reflectance(mat, float(w), cfg) for w in wl_dense]
    ax.plot(wl_dense, refs, linewidth=2, label=mat)

for wl in cfg.wavelengths:
    ax.axvline(wl, color='gray', linestyle=':', alpha=0.5)

ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Reflectance', fontsize=12)
ax.set_title('Chromasense Material Spectral Signatures', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.0)
fig.tight_layout()
plt.show()

**Key observations:**

- **Copper** has a very distinctive rising reflectance from blue to IR -- this is
  the classic copper colour.
- **Silicon** is relatively flat across all bands.
- **Mold compound** is dark across all wavelengths.
- **Oxidized copper** shows a suppressed version of the copper curve, making it
  spectrally distinguishable from clean copper.

In [ ]:
# Exact reflectance values at the Chromasense wavelengths
for mat in cfg.material_names:
    vals = [material_reflectance(mat, wl, cfg) for wl in cfg.wavelengths]
    print(f"{mat:20s}  {vals}")

---
## 2. Custom Configuration

You can define your own wavelength set and material spectra for different
Chromasense hardware configurations.

In [ ]:
custom_cfg = SpectralConfig(
    wavelengths=[400.0, 500.0, 600.0, 700.0, 900.0],
    material_spectra={
        'gold': {400.0: 0.35, 500.0: 0.45, 600.0: 0.85, 700.0: 0.95, 900.0: 0.97},
        'nickel': {400.0: 0.55, 500.0: 0.56, 600.0: 0.58, 700.0: 0.60, 900.0: 0.62},
    },
)
print(f"Custom channels: {custom_cfg.n_channels}")
print(f"Custom materials: {custom_cfg.material_names}")
print(f"Gold @ 550nm (interpolated): {material_reflectance('gold', 550.0, custom_cfg):.3f}")